In [2]:
!pip install torchcodec
!pip install --upgrade --quiet pip
!pip install --upgrade --quiet datasets[audio] transformers accelerate evaluate jiwer

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import json
import torch
import os
from pathlib import Path
from dataclasses import dataclass
from datasets import Dataset, DatasetDict, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate
import warnings
warnings.filterwarnings("ignore")

In [5]:
"""
WHISPER FINE-TUNING NOTEBOOK FOR COLAB FREE
Simplified approach matching the original notebook style
"""

import json
import torch
from pathlib import Path
from dataclasses import dataclass
from datasets import Dataset, DatasetDict, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate


# ============================================================================
# LOAD DATASET FROM JSON METADATA
# ============================================================================

def load_datasets_from_json(dataset_dir: str) -> DatasetDict:
    """Load train and eval datasets from JSON metadata files"""

    dataset_path = Path(dataset_dir)
    train_file = dataset_path / "train_metadata.json"
    eval_file = dataset_path / "eval_metadata.json"

    if not train_file.exists() or not eval_file.exists():
        raise FileNotFoundError(f"Missing JSON metadata files in {dataset_dir}")

    # Load JSON files
    def load_json(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    train_data = load_json(train_file)
    eval_data = load_json(eval_file)

    # Create datasets
    train_dataset = Dataset.from_dict({
        "audio": [d["audio_path"] for d in train_data],
        "transcript": [d["ground_truth"] for d in train_data],
    })

    eval_dataset = Dataset.from_dict({
        "audio": [d["audio_path"] for d in eval_data],
        "transcript": [d["ground_truth"] for d in eval_data],
    })

    # Cast to Audio type
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
    eval_dataset = eval_dataset.cast_column("audio", Audio(sampling_rate=16000))

    return DatasetDict({"train": train_dataset, "eval": eval_dataset})


# ============================================================================
# LOAD PROCESSOR
# ============================================================================

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny",
    language="French",
    task="transcribe"
)


# ============================================================================
# PREPARE DATA
# ============================================================================

def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids

    return batch


# ============================================================================
# DATA COLLATOR
# ============================================================================

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: any
    decoder_start_token_id: int

    def __call__(self, features):
        inputs = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(inputs, return_tensors="pt")

        labels = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(labels, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


# ============================================================================
# EVALUATION METRICS
# ============================================================================

metric = evaluate.load("wer")

def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    # Load dataset
    print("Loading datasets...")
    dataset_dir = "/content/drive/MyDrive/asr/output/whisper_children_dataset/training_dataset"
    datasets = load_datasets_from_json(dataset_dir)

    print(f"Train samples: {len(datasets['train'])}")
    print(f"Eval samples: {len(datasets['eval'])}\n")

    # Prepare data
    print("Preparing datasets...")
    datasets = datasets.map(
        prepare_dataset,
        remove_columns=datasets["train"].column_names,
        num_proc=2
    )

    # Load model
    print("Loading model...")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
    model.generation_config.language = "French"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None

    # Data collator
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    # Training arguments (optimized for Colab Free)
    training_args = Seq2SeqTrainingArguments(
        output_dir="/content/drive/MyDrive/asr/output/whisper_children_dataset/whisper_tiny_fr",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        num_train_epochs=1,
        max_steps=80,  # Limit steps to avoid memory issues
        save_steps=100,
        eval_steps=100,
        logging_steps=10,
        eval_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=False,
        report_to=[],
        fp16=False,
    )

    # Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=datasets["train"],
        eval_dataset=datasets["eval"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.tokenizer,
    )

    # Save processor
    processor.save_pretrained(training_args.output_dir)

    # Train
    print("\n" + "="*70)
    print("Starting training...")
    print("="*70 + "\n")
    trainer.train()

    print("\n" + "="*70)
    print("✅ Training complete!")
    print("="*70)

Loading datasets...
Train samples: 277
Eval samples: 70

Preparing datasets...


Map (num_proc=2):   0%|          | 0/277 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/70 [00:00<?, ? examples/s]

Loading model...


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Starting training...



Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss



✅ Training complete!


In [ ]:
    # ------------------------------------------------------------------------
    # Load trained model and evaluate (WER)
    # ------------------------------------------------------------------------
    print("\nLoading trained model for evaluation...")
    model = WhisperForConditionalGeneration.from_pretrained(
        "/content/drive/MyDrive/asr/output/whisper_children_dataset/whisper_tiny_fr/checkpoint-80"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        eval_dataset=datasets["eval"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.tokenizer,
    )

    print("\nRunning evaluation on trained model...")
    metrics = trainer.evaluate()
    print(f"WER: {metrics['eval_wer']:.2f}")



Loading trained model for evaluation...

Running evaluation on trained model...
